# Phase 14: frozen-prediction statistical analysis

All results are conditional on the existing, fixed out-of-fold predictions: 233 patients, 3064 slices, five patient-disjoint held-out folds, 11 models. A5 is the original full MorphoNet-BT model. No inference, model loading, training, checkpoint selection, split generation or preprocessing is performed.

Primary endpoint: patient accuracy, based on argmax of each patient's mean slice probabilities. Exact two-sided McNemar tests compare paired correctness, using Binomial(n_discordant, 0.5), including p=1 for no discordances. Holm correction controls the family of 10 unique comparisons (A4 vs A5/five baselines and four additional adjacent ablations). A5-A4 reverses A4-A5 and is displayed without adding a duplicate hypothesis. Differences are left minus right, in percentage points; AUC is multiplied by 100 only for presentation.

Unstratified, paired patient bootstrap: 10,000 shared draws of 233 patient IDs with replacement, NumPy default_rng seed 42. All models and both levels use the identical draws. Slice statistics retain all slices from each drawn patient, including multiplicities; they are slice-weighted estimands with patient clustering. Patient statistics weight each patient equally. Accuracy, three-class macro-F1 and macro one-vs-rest AUC are recalculated in every replicate. Weighted tied-score rank AUC is validated against sklearn and explicit repeated-cluster samples. Percentile 2.5/97.5 intervals use linear interpolation. AUC replicates missing any class are marked invalid, excluded from its percentile interval, and counted. Intervals are marginal, not multiplicity-adjusted; no p-values are inferred from bootstrap sign counts.

The exploratory Meningioma analysis conditions on the 82 truly Meningioma patients (708 slices), reports recall and paired patient/cluster recall intervals, and uses a separate 10-test Holm family for patient McNemar. Its shared 82-patient draws continue the same seeded random stream. Per-fold patient and slice accuracy differences are descriptive only; folds are not treated as five independent experimental replications.

Limitations: A4 was highlighted after observing Phase 13 outcomes; these are exploratory post-selection analyses, not preregistered confirmatory evidence. Holm does not repair outcome-driven selection or cover unknown prior comparisons. These conditional prediction analyses omit training/checkpoint randomness; overlapping cross-validation training sets induce dependence not captured by ordinary patient resampling or McNemar. Patient independence and representative cohort sampling remain assumptions. No external cohort or prospective clinical validation is supplied. Slice and patient endpoints differ in weighting; subgroup analysis has only 82 patients. Non-significance does not establish equivalence. Bootstrap intervals for F1/AUC are supplementary and unadjusted. Sequential effects measure each added component in that specific context, not independent causal contributions or a factorial interaction analysis. No new training experiments are performed.

References: [SciPy exact binomial test](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.binomtest.html), [McNemar exact-test definition](https://www.statsmodels.org/dev/generated/statsmodels.stats.contingency_tables.mcnemar.html), [Holm multiple testing](https://www.statsmodels.org/stable/generated/statsmodels.stats.multitest.multipletests.html), [paired bootstrap](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.bootstrap.html).


## Public-copy execution policy
This is an English rendering of a historical research notebook. Outputs and execution counts were cleared; no cell was rerun for Phase 18. Historical findings below describe earlier runs only. Every code cell is gated, including setup cells that may perform model checks, preprocessing or writes. Enable `CONFIRM_PUBLIC_WORKFLOW` deliberately in the first code cell only after reviewing [the reproduction guide](../docs/reproducibility.md). Existing stage-specific confirmations must also be enabled separately. Run from the repository root or set `MORPHONET_ROOT` to a private writable experiment workspace with the same source/config layout; set `MORPHONET_DATA_ROOT` for dataset discovery. Original source/configuration fingerprints, private split manifests and checkpoints are not portable artifacts; do not bypass their validation. See the guide for prerequisites and limitations.


In [ ]:
CONFIRM_PUBLIC_WORKFLOW = False  # Explicit opt-in; no cell has been rerun.
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
from pathlib import Path
import sys
ROOT = Path(__import__("os").environ.get("MORPHONET_ROOT", str(Path.cwd()))).resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
assert (ROOT / 'src/phase14_statistics.py').is_file()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.phase14_statistics import run, validate_synthetic, read
CONFIRM_STATISTICAL_ANALYSIS = False
OVERWRITE = False
RESUME_ANALYSIS = True
BOOTSTRAP_ITERATIONS = 10000
RANDOM_SEED = 42
SETTINGS = dict(CONFIRM_STATISTICAL_ANALYSIS=CONFIRM_STATISTICAL_ANALYSIS,
               OVERWRITE=OVERWRITE, RESUME_ANALYSIS=RESUME_ANALYSIS,
               BOOTSTRAP_ITERATIONS=BOOTSTRAP_ITERATIONS, RANDOM_SEED=RANDOM_SEED)
print('Authorized statistics only. No model training or inference.')

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
validate_synthetic()

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
OUTPUT = run(ROOT, SETTINGS)
print(OUTPUT)

## Complete results

Effect directions are left minus right. The A5-A4 row is the reverse display of the same A4-A5 hypothesis, not an additional test. CSV files retain full precision. Marginal confidence intervals do not define Holm significance.

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
import pandas as pd
from IPython.display import display, Markdown, Image
pd.set_option('display.max_rows', 200)
pd.set_option('display.max_columns', 20)
pd.set_option('display.precision', 10)
for name in ['model_metrics_with_ci', 'exact_mcnemar_holm', 'paired_effects_with_ci', 'meningioma_absolute_recall', 'meningioma_exact_mcnemar_holm', 'meningioma_paired_recall', 'fold_accuracy_differences', 'fold_robustness_summary']:
    display(Markdown('### ' + name.replace('_', ' ').title()))
    display(read(OUTPUT / (name + '.csv')))

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
for name in ['patient_paired_effects', 'slice_paired_effects', 'meningioma_paired_recall', 'fold_robustness']:
    display(Image(filename=str(OUTPUT / 'figures' / (name + '.png'))))

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
import json
audit = json.loads((OUTPUT / 'final_scientific_audit.json').read_text())
assert audit['status'] == 'PASS'
assert audit['training_runs'] == 0 and audit['optimizer_steps'] == 0
assert audit['frozen_artifacts_unchanged']
display(audit)
print('Phase 14 completed. STOP before Phase 15.')